# Urban Heat Island (UHI) Classification — Random Forest Baseline

A streamlined modelling notebook that:

* Uses **pre-generated GeoTIFF files** (indices) and **pre-computed building
  density GeoParquet files** stored on Google Drive.  No on-the-fly shapefile
  spatial joins; no Planetary Computer calls.
* Trains a **single Random Forest classifier** on the combined features.
* Predicts UHI class on a held-out validation region and writes a submission
  CSV.

The notebook uses **generic names** for objects (`train_*`, `val_*`) rather
than city-specific ones so the same cells can be reused for any training /
validation region by swapping the input paths in the configuration cell.

In [ ]:
# ---------------------------------------------------------------------------
# Clone the bc-II repository for CSV data files (UHI labels, validation set)
# ---------------------------------------------------------------------------
!git clone https://github.com/chase-kusterer/bc-II.git
%cd bc-II
!cat .env_bin.txt > /dev/null
!pip install -r /content/bc-II/requirements.txt

In [ ]:
# ---------------------------------------------------------------------------
# Mount Google Drive — source of GeoTIFF indices files and cached building
# density GeoParquet files (both produced by the upstream notebooks).
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------------------
# Dependencies
# ---------------------------------------------------------------------------
!pip install rioxarray

In [ ]:
# Suppress non-critical warnings
import warnings
warnings.filterwarnings('ignore')

# File management
import os

# Data manipulation
import numpy as np
import pandas as pd

# Geospatial raster / vector I/O
import rioxarray as rxr
import rasterio
import geopandas as gpd
from pyproj import Proj, Transformer

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling — Random Forest only; LabelEncoder and StandardScaler removed
# because RF accepts string labels and is scale-invariant.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

# Progress bar for loops
from tqdm import tqdm

## Configuration

All region-specific paths live here.  Point `TRAIN_*` at your training region
and `VAL_*` at the validation region.  The rest of the notebook is
region-agnostic.

In [ ]:
# ===========================================================================
# TRAINING REGION — inputs used to fit the classifier
# ===========================================================================
TRAIN_UHI_CSV       = "/content/bc-II/Data/sample_chile_uhi_data.csv"
TRAIN_GEOTIFF_PATH  = "/content/drive/MyDrive/Business Challenge II/GeoTIFFs/" \
                      "Santiago_MBR_Best_Single(2024-06-11).tiff"
TRAIN_DENSITY_CACHE = "/content/drive/MyDrive/Business Challenge II/" \
                      "BuildingDensity/Santiago_building_density_100m.parquet"

# ===========================================================================
# VALIDATION REGION — held-out area the trained model is applied to
# ===========================================================================
VAL_UHI_CSV       = "/content/bc-II/Data/validation_dataset.csv"
VAL_GEOTIFF_PATH  = "/content/drive/MyDrive/Business Challenge II/GeoTIFFs/" \
                    "Freetown_TBR_Median_Composite(2023-12-15TO2024-02-15).tiff"
VAL_DENSITY_CACHE = "/content/drive/MyDrive/Business Challenge II/" \
                    "BuildingDensity/Freetown_building_density_100m.parquet"

# ===========================================================================
# Output location for the submission CSV
# ===========================================================================
SUBMISSION_CSV = "/content/Predicted_Dataset.csv"

# Sanity-check that all input files exist before running the long pipeline
for p in [TRAIN_UHI_CSV, TRAIN_GEOTIFF_PATH, TRAIN_DENSITY_CACHE,
          VAL_UHI_CSV, VAL_GEOTIFF_PATH, VAL_DENSITY_CACHE]:
    assert os.path.exists(p), f"Missing input file: {p}"
print("All input files are present.")

## Helper Functions

Two small utilities used by both the training and validation pipelines:

* `extract_band_values` — unchanged from the original notebook; reads NDVI /
  NDBI / NDWI bands from a GeoTIFF at each lat/lon in a CSV.
* `load_building_density_for_concat` — loads a cached GeoParquet and returns
  just the `building_density_100m` column so it can be safely concatenated
  (`axis=1`) with a band-values frame that already contains all the original
  CSV columns.

In [ ]:
# ===========================================================================
# Extract NDVI / NDBI / NDWI at each CSV point from a 3-band indices GeoTIFF
# ===========================================================================
def extract_band_values(geotiff_path, csv_input_path):
    """
    Reads per-point values of the three bands (1=NDVI, 2=NDBI, 3=NDWI) from
    `geotiff_path` at every (Longitude, Latitude) in `csv_input_path`.

    Returns the CSV as a DataFrame with three additional columns:
    median_NDVI, median_NDBI, median_NDWI.

    Assumes the GeoTIFF is in EPSG:4326 (same CRS as the CSV's lat/lon).
    """
    # Load the GeoTIFF as a lazy xarray DataArray
    data = rxr.open_rasterio(geotiff_path)

    # Read CSV and pull out coordinates
    df = pd.read_csv(csv_input_path)
    latitudes  = df['Latitude'].values
    longitudes = df['Longitude'].values

    # Accumulators for per-point band values
    ndvi_list, ndbi_list, ndwi_list = [], [], []

    for lat, lon in tqdm(
        zip(latitudes, longitudes),
        total=len(latitudes),
        desc="Mapping values",
    ):
        # `method="nearest"` snaps the query point to the closest pixel
        ndvi_list.append(data.sel(x=lon, y=lat, band=1, method="nearest").values)
        ndbi_list.append(data.sel(x=lon, y=lat, band=2, method="nearest").values)
        ndwi_list.append(data.sel(x=lon, y=lat, band=3, method="nearest").values)

    df["median_NDVI"] = ndvi_list
    df["median_NDBI"] = ndbi_list
    df["median_NDWI"] = ndwi_list
    return df


# ===========================================================================
# Load a cached building-density GeoParquet for axis=1 concat
# ===========================================================================
def load_building_density_for_concat(cache_path):
    """
    Load the cached building-density GeoParquet produced by the shpFileReader
    notebook and return ONLY the `building_density_100m` column as a plain
    DataFrame.  This avoids duplicating Latitude/Longitude/geometry columns
    that are already present in the band-values frame.

    Row order is preserved from the original CSV, so the output is safe to
    concat on axis=1 with the band-values frame.
    """
    gdf = gpd.read_parquet(cache_path)
    return pd.DataFrame(gdf[['building_density_100m']])

## 1. Build the Training DataFrame

Three steps:

1. Load ground-truth UHI labels from CSV.
2. Extract NDVI / NDBI / NDWI at each point from the training GeoTIFF.
3. Load building density from the cached GeoParquet.
4. Concatenate on axis=1 and select modelling columns.

In [ ]:
# ---------------------------------------------------------------------------
# Quick look at the training ground-truth CSV
# ---------------------------------------------------------------------------
train_uhi_df = pd.read_csv(TRAIN_UHI_CSV)
print(f"Training points: {len(train_uhi_df)}")
train_uhi_df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Extract indices from the training GeoTIFF at each labelled point
# ---------------------------------------------------------------------------
train_band_data = extract_band_values(
    geotiff_path=TRAIN_GEOTIFF_PATH,
    csv_input_path=TRAIN_UHI_CSV,
)
train_band_data.head()

In [ ]:
# ---------------------------------------------------------------------------
# Load the cached training-region building density
# (produced by the shpFileReader notebook — no recomputation here)
# ---------------------------------------------------------------------------
train_density_data = load_building_density_for_concat(TRAIN_DENSITY_CACHE)
print(f"Training density rows: {len(train_density_data)}")
train_density_data.head()

In [ ]:
# ---------------------------------------------------------------------------
# Combine labels + spectral indices + building density into a single frame.
# Both inputs come from the same CSV read in the same order, so positional
# axis=1 concat aligns rows correctly.
# ---------------------------------------------------------------------------
train_combined = pd.concat(
    [
        train_band_data[['Longitude', 'Latitude', 'UHI_Class',
                         'median_NDVI', 'median_NDBI', 'median_NDWI']]
            .reset_index(drop=True),
        train_density_data.reset_index(drop=True),
    ],
    axis=1,
)
print(f"Combined training shape: {train_combined.shape}")
train_combined.head()

### Remove Duplicates

Some (lat, lon) pairs can map to the same pixel in the GeoTIFF, yielding rows
with identical feature values.  Training on duplicates biases the classifier,
so we drop exact duplicates on the four predictor columns.

The `tuple(...)` step converts any numpy-array cells (produced by
`data.sel().values`) into hashable tuples so pandas can compare them.

In [ ]:
# ---------------------------------------------------------------------------
# Drop duplicate rows based on the four predictor columns
# ---------------------------------------------------------------------------
cols_to_check = ['median_NDVI', 'median_NDBI', 'median_NDWI', 'building_density_100m']

# Make numpy-array cells hashable so duplicated() can compare them
for col in cols_to_check:
    train_combined[col] = train_combined[col].apply(
        lambda x: tuple(x) if isinstance(x, np.ndarray) and x.ndim > 0 else x
    )

n_dupes = train_combined.duplicated(subset=cols_to_check).sum()
print(f"Duplicate rows based on {cols_to_check}: {n_dupes}")

train_data = (
    train_combined
    .drop_duplicates(subset=cols_to_check, keep='first')
    .reset_index(drop=True)
)
print(f"Training shape after dedup: {train_data.shape}")
train_data.head()

## 2. Prepare Features and Split

Only the four predictor columns and the target (`UHI_Class`) are kept.
Latitude / Longitude are excluded — they are coordinates, not predictive
features in their own right.

In [ ]:
# ---------------------------------------------------------------------------
# Select predictor columns + target
# ---------------------------------------------------------------------------
feature_cols = ['median_NDVI', 'median_NDBI', 'median_NDWI', 'building_density_100m']
target_col   = 'UHI_Class'

train_df = train_data[feature_cols + [target_col]].copy()
train_df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Train / test split — stratified to preserve class proportions.
# Random Forest handles string labels directly, so no LabelEncoder is needed.
# ---------------------------------------------------------------------------
X = train_df[feature_cols].values
y = train_df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=123
)
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## 3. Train the Random Forest

`class_weight='balanced'` compensates for class imbalance without needing
oversampling methods like SMOTE.  Feature scaling is not applied because
Random Forest (a tree-based model) is scale-invariant.

In [ ]:
# ---------------------------------------------------------------------------
# Instantiate and train the Random Forest classifier
# ---------------------------------------------------------------------------
rf_model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight='balanced',
)
rf_model.fit(X_train, y_train)
print("Random Forest training complete.")

## 4. Out-of-Sample Evaluation

In [ ]:
# ---------------------------------------------------------------------------
# Predict on the held-out test split and report accuracy + per-class metrics
# ---------------------------------------------------------------------------
outsample_predictions = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, outsample_predictions)
print(f"Accuracy: {accuracy:.4f}\n")
print(classification_report(y_test, outsample_predictions))

In [ ]:
# ---------------------------------------------------------------------------
# Confusion matrix — visualises where the model confuses classes
# ---------------------------------------------------------------------------
test_labels = np.unique(y_test)
cm = confusion_matrix(y_test, outsample_predictions, labels=test_labels)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_labels)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.grid(False)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Feature importance — which predictors drove the Random Forest's decisions
# ---------------------------------------------------------------------------
importances = rf_model.feature_importances_
order = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.title("Feature Importances")
plt.bar(range(len(importances)), importances[order], align='center')
plt.xticks(range(len(importances)),
           [feature_cols[i] for i in order], rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Apply the Model to the Validation Region

Same pipeline: extract indices from the validation GeoTIFF, load cached
building density, combine, predict.

In [ ]:
# ---------------------------------------------------------------------------
# Load the validation point list
# ---------------------------------------------------------------------------
val_uhi_df = pd.read_csv(VAL_UHI_CSV)
print(f"Validation points: {len(val_uhi_df)}")
val_uhi_df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Extract indices for validation points
# ---------------------------------------------------------------------------
val_band_data = extract_band_values(
    geotiff_path=VAL_GEOTIFF_PATH,
    csv_input_path=VAL_UHI_CSV,
)
val_band_data.head()

In [ ]:
# ---------------------------------------------------------------------------
# Load cached validation-region building density
# ---------------------------------------------------------------------------
val_density_data = load_building_density_for_concat(VAL_DENSITY_CACHE)
val_density_data.head()

In [ ]:
# ---------------------------------------------------------------------------
# Combine into the validation modelling frame.  Keep Lat/Lon because they are
# needed for the submission file.
# ---------------------------------------------------------------------------
val_combined = pd.concat(
    [
        val_band_data[['Longitude', 'Latitude',
                       'median_NDVI', 'median_NDBI', 'median_NDWI']]
            .reset_index(drop=True),
        val_density_data.reset_index(drop=True),
    ],
    axis=1,
)
print(f"Validation combined shape: {val_combined.shape}")
val_combined.head()

In [ ]:
# ---------------------------------------------------------------------------
# Normalise numpy-array cells (same treatment as training) so the predictor
# matrix is a clean 2-D array of scalars.
# ---------------------------------------------------------------------------
for col in feature_cols:
    val_combined[col] = val_combined[col].apply(
        lambda x: x.item() if isinstance(x, np.ndarray) and x.ndim > 0 else x
    )

X_val = val_combined[feature_cols].values
print(f"X_val shape: {X_val.shape}")

## 6. Predict and Save the Submission

In [ ]:
# ---------------------------------------------------------------------------
# Predict UHI class for each validation point
# ---------------------------------------------------------------------------
val_predictions = rf_model.predict(X_val)

# Build the submission DataFrame with coordinates + predicted class
submission_df = pd.DataFrame({
    'Longitude' : val_combined['Longitude'].values,
    'Latitude'  : val_combined['Latitude'].values,
    'UHI_Class' : val_predictions,
})

print("Predicted class distribution:")
print(submission_df['UHI_Class'].value_counts())
submission_df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Save the submission CSV to Colab local storage (outside the cloned repo)
# ---------------------------------------------------------------------------
submission_df.to_csv(SUBMISSION_CSV, index=False)
print(f"Submission saved to: {SUBMISSION_CSV}")

## Conclusion

Once you are satisfied with the model's performance, share
`Predicted_Dataset.csv` with your instructor for evaluation.

Next steps you might consider:

* Swap the training GeoTIFF for a median-composite file instead of a single
  best-single-scene file.
* Add more training regions by extending the configuration cell with
  additional CSVs + GeoTIFF + GeoParquet paths and concatenating the resulting
  `train_combined` frames before the dedup step.
* Tune `n_estimators`, `max_depth`, `min_samples_leaf`, etc. via grid search.